# Building Ethical Vision Systems
## Part 1 — Computer Vision Foundations Through the Lens of an Audit

**Ghana Data Science Summit 2026 · Ho, Ghana**
*Tutorial lead: Nana Sam Yeboah*

---

A health startup in Accra has built a skin condition detection app for community health workers in rural Ghana. Before it goes live, we have been asked to audit it.

This notebook is the technical foundation for that audit. It follows the live tutorial led by Sam. You can modify any parameter, re-run any cell, and follow any of the curated further-reading links to go as deep as you want on any concept.




---
##  SECTION 00 — SETUP
*Installing dependencies, loading libraries, configuring the environment.*

---


In [ ]:
# Install all dependencies for Part 1
%pip install -q torch torchvision timm opencv-python-headless pillow numpy matplotlib seaborn scikit-learn ultralytics transformers fairlearn scipy gdown tqdm
print("✓ Dependencies installed")


In [ ]:
# All imports for the full notebook
import os, sys, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from pathlib import Path
import torch
import torch.nn as nn
import torchvision.transforms as T
from tqdm.auto import tqdm

# Add project root to path so utils imports work
sys.path.insert(0, str(Path().resolve().parent))

print(f"PyTorch   : {torch.__version__}")
print(f"OpenCV    : {cv2.__version__}")
print(f"NumPy     : {np.__version__}")


In [ ]:
# Configuration — edit DEVICE if you have a GPU available
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
SEED       = 42
DATA_DIR   = Path("../data/sample_images")
ASSETS_DIR = Path("../assets")

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Device: {DEVICE}")
print(f"Data directory: {DATA_DIR.resolve()}")


In [ ]:
# Download sample images (hosted on Google Drive)
# If this cell fails, see data/README.md for alternative download instructions

import gdown

DATA_DIR.mkdir(parents=True, exist_ok=True)

# Publicly shared folder with 5 sample skin images
FOLDER_URL = "https://drive.google.com/drive/folders/17aTozdEucMpuuF3k23GH6X4IVtFCdxvE"

try:
    gdown.download_folder(FOLDER_URL, output=str(DATA_DIR), quiet=False)
    print("✓ Sample images downloaded")
    SAMPLE_IMAGE_PATH="../data/sample_images/eczema_statis_dermatitis.png"
except Exception as e:
    print(f"Auto-download failed ({e}). See data/README.md for manual instructions.")



### What is Computer Vision?

**Computer vision** is the field of teaching machines to interpret and understand digital images and video. The name is aspirational — "vision" suggests something close to human perception, but the underlying process is radically different. Where a human looks at a skin lesion and integrates colour, texture, shape, and clinical context into a judgment, a computer vision model computes a sequence of mathematical transformations on a grid of numbers.

The fundamental challenge is this: the same object looks different in every image. A melanoma photographed under a dermatoscope in Vienna looks nothing like the same condition photographed with an Android phone in Brong-Ahafo. Shadows change. Angles change. Skin tone changes. The pixel values are different, but the object is the same. Teaching a model to understand this — to generalise beyond the exact images it was trained on — is the core technical problem.

The field spans every domain where machines need to interpret visual data: radiology and pathology (finding tumours in scans and slides), autonomous vehicles (detecting pedestrians and lane markings in real time), satellite imagery (mapping deforestation or flood extent), agricultural inspection (detecting disease in crops from drone footage), and the application at the centre of this tutorial — skin condition detection in low-resource clinical settings.

The image we are working with is formally represented as a three-dimensional tensor:

$$I \in \mathbb{R}^{H \times W \times C}$$

where $H$ is height in pixels, $W$ is width in pixels, and $C$ is the number of channels (3 for RGB colour images, 1 for grayscale). Every element of this tensor is an integer from 0 to 255 representing light intensity. The model sees only these numbers.

**The African context.** Most benchmark datasets in computer vision were built in North America, Europe, and East Asia. This is not incidental because data collection requires resources, annotation requires expertise, and both have been concentrated in places with established research infrastructure. The performance gap on African imagery is real, documented, and underaddressed. This tutorial is grounded in that gap.

---
📚 **Further Reading & Watching**

| Resource | Type | Why it's worth your time |
|---|---|---|
| [How Computers See Images — Computerphile](https://www.youtube.com/watch?v=15aqFQQVBWU) | 📹 Video (6 min) | Clear, accessible explanation of how images are stored as numbers |
| [OpenCV Python Tutorials](https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html) | 📖 Docs | Official reference  |
| [Colour Spaces in OpenCV — LearnOpenCV](https://learnopencv.com/color-spaces-in-opencv-cpp-python/) | 📄 Article | Practical guide to RGB, BGR, HSV, and when to use each |

---


---
## SECTION 01 — THE IMAGE ARRIVES
*Story beat: A community health worker photographs a patient's skin lesion. The image has problems.*

---


A community health worker in Brong-Ahafo has just sent her first image. She photographed the patient's arm outdoors, midday sun, with her Tecno Spark Android. The image is in our hands now. Before any model sees it, we need to understand what it actually is — not in clinical terms, but in mathematical ones. What does this image look like to a computer?


### Digital Images as Tensors

A digital image is a three-dimensional array of numbers. This is the single most important thing to understand about computer vision: **every image is a tensor**, and every model operates entirely on that tensor. There are no pixels in the deep sense — no colours, no shapes, no objects. There are only numbers.

**Pixel values** are integers from 0 to 255 per channel, where 0 means no light (black) and 255 means maximum intensity of that channel's colour. A red pixel with full saturation is $[255, 0, 0]$ in RGB. A grey pixel at half brightness is $[128, 128, 128]$.

$$\text{pixel}_{(i,j)} = [R_{(i,j)},\; G_{(i,j)},\; B_{(i,j)}] \in [0,\,255]^3$$

**RGB vs BGR.** RGB is the standard: Red first, then Green, then Blue. OpenCV, confusingly, defaults to BGR — Blue first. This is a notorious source of bugs. If your colours look wrong, you have almost certainly loaded an image in BGR and displayed it as RGB. Every function in our `image_utils.py` accepts and returns RGB to avoid this confusion.

**Colour spaces** are different mathematical representations of colour information:
- **RGB**: the standard. Each channel encodes one primary colour. Intuitive for display.
- **HSV (Hue · Saturation · Value)**: separates *what colour* (hue) from *how vivid* (saturation) and *how bright* (value). Critical advantage: the H channel is nearly constant under lighting changes. A blue lesion is blue whether lit by noon sun or afternoon shade.
- **Grayscale**: a single channel representing luminance. Loses colour information but reduces data by 3×. Used as input for many edge detection and morphological operations.

For skin imaging, HSV deserves special attention. The V channel encodes brightness variation caused by lighting — exactly the information we want to *remove* to make the model robust to varying phone-camera conditions. Separating it out gives us a handle on the problem.


In [ ]:
# Load the narrative image and inspect it
from utils.image_utils import load_image

img_rgb  = load_image(SAMPLE_IMAGE_PATH, colour_space="rgb")
img_bgr  = load_image(SAMPLE_IMAGE_PATH, colour_space="bgr")
img_hsv  = load_image(SAMPLE_IMAGE_PATH, colour_space="hsv")
img_gray = load_image(SAMPLE_IMAGE_PATH, colour_space="gray")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img_rgb);         axes[0].set_title("RGB (correct)")
axes[1].imshow(img_bgr);         axes[1].set_title("BGR displayed as RGB (wrong colours)")
axes[2].imshow(img_gray, cmap="gray"); axes[2].set_title("Grayscale")
axes[3].imshow(img_hsv);         axes[3].set_title("HSV (displayed raw — not perceptually correct)")
for ax in axes: ax.axis("off")
plt.suptitle("The same image in different colour representations", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Show individual RGB channels
r, g, b = cv2.split(img_rgb)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img_rgb);              axes[0].set_title("Original RGB")
axes[1].imshow(r, cmap="Reds_r");    axes[1].set_title("Red channel")
axes[2].imshow(g, cmap="Greens_r");  axes[2].set_title("Green channel")
axes[3].imshow(b, cmap="Blues_r");   axes[3].set_title("Blue channel")
for ax in axes: ax.axis("off")
plt.suptitle("RGB channel decomposition", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Show HSV channels — H separates colour from brightness
h, s, v = cv2.split(img_hsv)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img_rgb);              axes[0].set_title("Original RGB")
axes[1].imshow(h, cmap="hsv");       axes[1].set_title("H — Hue (colour type)")
axes[2].imshow(s, cmap="gray");      axes[2].set_title("S — Saturation (colour intensity)")
axes[3].imshow(v, cmap="gray");      axes[3].set_title("V — Value (brightness)")
for ax in axes: ax.axis("off")
plt.suptitle("HSV decomposition — V channel reveals the lighting problem", fontsize=12)
plt.tight_layout()
plt.show()
print("Notice: the V channel shows strong variation across the lesion — uneven illumination")


In [ ]:
# Show a 5×5 pixel crop as raw numbers — this is what the model actually sees
CROP_ROW, CROP_COL, CROP_SIZE = 100, 100, 5
crop = img_rgb[CROP_ROW:CROP_ROW+CROP_SIZE, CROP_COL:CROP_COL+CROP_SIZE]

print(f"Pixel values at rows {CROP_ROW}–{CROP_ROW+CROP_SIZE-1}, "
      f"cols {CROP_COL}–{CROP_COL+CROP_SIZE-1}:")
print("(Each row is [R, G, B])")
print(crop)


In [ ]:
# 🧪 EXPERIMENT — try changing these values
CROP_ROW  = 150  # try a different row
CROP_COL  = 200  # try a different column
CROP_SIZE = 5    # try 3 or 10

crop = img_rgb[CROP_ROW:CROP_ROW+CROP_SIZE, CROP_COL:CROP_COL+CROP_SIZE]
print(f"Pixel values at ({CROP_ROW}, {CROP_COL}):")
print(crop)
# Notice: inside the lesion, RGB values are lower across all channels.
# The difference between lesion and healthy skin is the signal the model must detect.


In [ ]:
# Pixel intensity histogram — the lighting problem made visible
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

colours = ["#e74c3c", "#2ecc71", "#3498db"]
channel_names = ["Red", "Green", "Blue"]

for i, (ch_name, colour) in enumerate(zip(channel_names, colours)):
    channel = img_rgb[:, :, i].ravel()
    axes[i].hist(channel, bins=64, color=colour, alpha=0.75, edgecolor="none")
    axes[i].axvline(channel.mean(), color="black", linestyle="--", linewidth=1.5,
                    label=f"mean={channel.mean():.1f}")
    axes[i].set_title(f"{ch_name} channel histogram")
    axes[i].set_xlabel("Pixel intensity (0–255)")
    axes[i].set_ylabel("Count")
    axes[i].legend()

plt.suptitle("Intensity histograms — values cluster in the dark range", fontsize=12)
plt.tight_layout()
plt.show()
print("The histogram peak sits left of 128 on all channels — the image is underexposed.")


### The Three Problems With This Image

Now that we can read the image as a mathematical object, we can describe precisely what is wrong with it — and why these problems matter for the model.

**1. Uneven lighting and low contrast.** The intensity histograms show a wide spread across the 0–255 range, with channel means sitting near or above the midpoint (128). There is uneven illumination: the V channel (brightness) varies significantly across the lesion, meaning some regions are brightly lit while others fall into shadow. Tall, narrow spikes in the histogram correspond to large uniform-colour regions (e.g. background), while the broad spread shows the lesion and skin occupy overlapping intensity ranges. Any classifier trained on consistently lit clinical images will struggle with this variability.

**2. Motion blur.** A handheld shot on a phone camera introduces blur from micro-vibrations during the exposure. Mathematically, the observed image $I_{obs}$ is the convolution of the true image $I_{true}$ with a point spread function (PSF): $I_{obs} = I_{true} * k$. The PSF for motion blur is a directional kernel. Reversing this(deconvolution) is an ill-posed problem because there are many possible sharp images that could have produced the same blurry result, so there's no single clean answer. In practice, we use an unsharp mask to recover approximate sharpness at the cost of amplifying noise.

**3. JPEG compression artefacts.** JPEG compresses images in 8×8 pixel blocks using the discrete cosine transform (DCT). At lower quality settings, sharp discontinuities appear at block boundaries — artificial high-frequency edges that are not part of the lesion. An edge detector will find these boundaries and potentially mistake them for lesion borders.

These are not edge cases. Every image arriving from a community health worker's phone will have some combination of these problems. The preprocessing pipeline we build in Section 02 exists to address them — and every decision in that pipeline encodes an assumption about the image distribution.

> ⚖️ **Ethical Lens:** The problems listed above are more severe on images taken in low-resource settings: outdoor lighting, consumer phones, no standardised photo protocol. The model's training data will not reflect these conditions. The gap between training distribution and deployment distribution is one of the most common reasons AI medical tools fail in the field.

---
📚 **Further Reading & Watching**

| Resource | Type | Why it's worth your time |
|---|---|---|
| [Histogram Equalisation — OpenCV Docs](https://docs.opencv.org/4.x/d5/daf/tutorial_py_histogram_equalization.html) | 📖 Docs | Official tutorial with code; covers CLAHE (the better version) too |

---


##  SECTION 02 — PREPROCESSING: MAKING THE IMAGE MODEL-READY

---


The image cannot go into the model as-is. The histogram is skewed, the edges are blurry, and JPEG artefacts litter the background. We need to build a preprocessing pipeline that corrects these problems before the model ever sees the image.


### Histogram Equalisation

Imagine the image's brightness values are furniture crammed into one dark corner of a room. Histogram equalisation moves the furniture around so it fills the whole room evenly — spreading the intensity values across the full 0–255 range so that fine detail hidden in the shadows becomes visible.

Formally, let $CDF(v)$ be the cumulative distribution function of pixel values. The equalised value for a pixel $v$ is:

$$T(v) = \left\lfloor \frac{CDF(v) - CDF_{\min}}{1 - CDF_{\min}} \times 255 \right\rfloor$$

**The limitation of global equalisation:** applying this transformation uniformly to the whole image can over-amplify noise in already-uniform regions (like a large patch of healthy skin). The artefacts this introduces can look like texture the model has never seen — and may confuse the classifier.

**CLAHE (Contrast Limited Adaptive Histogram Equalisation):** the solution practitioners actually use. The image is divided into small tiles (default 8×8 pixels). Each tile is equalised independently. A clip limit prevents any single intensity bin from accumulating too large a count — this suppresses the noise amplification problem. The tile boundaries are blended to avoid blocking artefacts.

**Why this matters for dark skin.** On Fitzpatrick Type V–VI skin, the contrast between a lesion and surrounding healthy tissue is typically lower than on lighter skin — the intensity values are closer together. Histogram equalisation is both more important (the signal is subtle) and more dangerous (over-equalisation can introduce artefacts that look like texture boundaries). CLAHE's clip limit is the dial that controls this trade-off, and it was almost certainly tuned on images that are not representative of dark skin.

> ⚖️ **Ethical Lens:** CLAHE parameters — clip limit and tile size — are chosen to produce "good" results on a validation set. If the validation set does not include Fitzpatrick Type V–VI images, the chosen parameters are optimised for lighter skin by default.


In [ ]:
from utils.image_utils import histogram_equalise

eq_global = histogram_equalise(img_rgb, method="global")
eq_clahe  = histogram_equalise(img_rgb, method="clahe", clip_limit=2.0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, image, title in zip(axes,
    [img_rgb, eq_global, eq_clahe],
    ["Original", "Global equalisation", "CLAHE (clip=2.0)"]):
    ax.imshow(image)
    ax.set_title(title, fontsize=12)
    ax.axis("off")
plt.suptitle("Histogram equalisation comparison", fontsize=13)
plt.tight_layout()
plt.show()


### Edge Detection

Edge detection is the bridge between image processing and machine understanding. An **edge** is a location where pixel intensity changes rapidly — the boundary between the lesion and healthy skin, the fold in the fabric of the garment, the outline of a vehicle. Detecting these boundaries is the first step toward understanding the shape and extent of what is in the image.

**The gradient intuition:** intensity change is measured as a spatial derivative. Where the derivative is large, intensity is changing fast — this is an edge. Where the derivative is near zero, intensity is constant — this is a flat region.

**The Sobel operator** computes this derivative by convolving the image with two 3×3 kernels:

$$K_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix}, \qquad K_y = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ 1 & 2 & 1 \end{bmatrix}$$

$K_x$ detects vertical edges; $K_y$ detects horizontal edges. The gradient magnitude is:
$$G = \sqrt{G_x^2 + G_y^2}$$

**The Canny edge detector** is the gold standard for lesion boundary detection. It has four stages:

1. **Gaussian smoothing** — blur slightly to suppress noise before computing gradients
2. **Sobel gradient computation** — compute $G_x$, $G_y$, and $G$
3. **Non-maximum suppression** — thin edges to 1-pixel width by keeping only local gradient maxima along the gradient direction
4. **Hysteresis thresholding** — two thresholds (low, high). Strong edges (above high) are kept unconditionally. Weak edges (between low and high) are kept only if they connect to a strong edge. Below-low pixels are discarded.

The output is a clean, thin, binary edge map.

**The bias.** Here is the problem. The contrast difference between a lesion and surrounding healthy tissue is significantly lower on darker skin tones. On Fitzpatrick Type V–VI skin, the lesion boundary in intensity values is subtler. The same Canny thresholds that produce complete, clean edge maps on lighter skin will produce broken, incomplete edges on darker skin — because the gradient magnitude at the lesion boundary is genuinely smaller.

This is not a theoretical concern. It is directly demonstrable, as the next three cells show.

> ⚖️ **Ethical Lens:** Canny thresholds are almost always chosen empirically on a validation set. If that validation set is not demographically representative, the thresholds are optimised for the overrepresented group. This is a preprocessing bias — it enters the model *before training even begins*.

---
📚 **Further Reading & Watching**

| Resource | Type | Why it's worth your time |
|---|---|---|
| [Edge Detection — Computerphile](https://www.youtube.com/watch?v=uihBwtPIBxM) | 📹 Video (8 min) | Excellent visual walkthrough of how edge detection actually works |
| [Edge Detection Using OpenCV — LearnOpenCV](https://learnopencv.com/edge-detection-using-opencv/) | 📄 Article | Side-by-side comparison of Sobel, Laplacian, and Canny with code |
| [A Computational Approach to Edge Detection — Canny (1986)](https://ieeexplore.ieee.org/document/4767851) | 📄 Paper | The original paper. Surprisingly readable. |

---


In [ ]:
from utils.image_utils import detect_edges, sharpen

img_gray_sharp = sharpen(img_gray)
# img_gray_sharp = cv2.cvtColor(sharpened_unsharp, cv2.COLOR_RGB2GRAY)
# img_gray_sharp = img_gray

sobel_x   = detect_edges(img_gray_sharp, method="sobel_x")
sobel_y   = detect_edges(img_gray_sharp, method="sobel_y")
sobel_mag = detect_edges(img_gray_sharp, method="sobel_magnitude")
canny     = detect_edges(img_gray_sharp, method="canny", low=50, high=150)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, img, title in zip(axes,[img_rgb, sobel_x, sobel_y, sobel_mag, canny],
    ["Original", "Sobel X (vertical edges)", "Sobel Y (horizontal edges)", "Sobel magnitude", "Canny (low=50, high=150)"]):
    ax.imshow(img, cmap="gray" if img.ndim == 2 else None)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.suptitle("Edge detection methods — Sobel and Canny", fontsize=12)
plt.tight_layout()
plt.show()


### Contour Detection and the Full Pipeline

Once we have an edge map, we can extract **contours** — continuous curves that represent the boundary of objects in the image. OpenCV's `findContours` traces connected edge pixels and returns them as polygon approximations. From the largest contour (which we assume is the lesion), we can compute:

- **Area** $A$: total pixel count inside the contour
- **Perimeter** $P$: total length of the contour boundary
- **Circularity** $C$: how close the shape is to a perfect circle

$$C = \frac{4\pi A}{P^2}$$

where $C = 1.0$ is a perfect circle and $C < 1$ indicates irregular borders. This is directly related to the **B criterion in the ABCDE rule** for melanoma detection (Border irregularity is a warning sign). Computational circularity operationalises a clinical judgment.

The full preprocessing pipeline chains all of these operations:

$$\text{raw image} \rightarrow \text{CLAHE} \rightarrow \text{shadow removal} \rightarrow \text{sharpening} \rightarrow \text{edge detection} \rightarrow \text{contour} \rightarrow \text{crop to ROI} \rightarrow \text{resize}(224 \times 224)$$

The final 224×224 image is what enters the model.

In [ ]:
from utils.image_utils import find_lesion_contour, crop_to_lesion, preprocess_pipeline, remove_shadow

# Run the full pipeline on the narrative image
img_preprocessed = preprocess_pipeline(img_rgb, target_size=(224, 224))

# Also get the intermediate steps for display
eq   = histogram_equalise(img_rgb, method="clahe")
ns   = remove_shadow(eq)
sh   = sharpen(ns, method="unsharp", alpha=0.5)
gr   = cv2.cvtColor(sh, cv2.COLOR_RGB2GRAY)
ed   = detect_edges(gr, method="canny", low=50, high=150)
contour, bounding_rect, area, perimeter, circularity = find_lesion_contour(ed)
cr   = crop_to_lesion(img_rgb, bounding_rect, padding=10)
rs   = cv2.resize(cr, (224, 224), interpolation=cv2.INTER_LANCZOS4)

pipeline_images = [img_rgb, eq, ns, sh, ed, rs]
pipeline_titles = ["1. Raw", "2. CLAHE", "3. Shadow removal",
                   "4. Sharpen", "5. Canny edges", "6. Crop & resize 224×224"]

fig, axes = plt.subplots(1, 6, figsize=(22, 4))
for ax, image, title in zip(axes, pipeline_images, pipeline_titles):
    ax.imshow(image, cmap="gray" if image.ndim == 2 else None)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.suptitle("Full preprocessing pipeline — raw image to model input", fontsize=12)
plt.tight_layout()
plt.show()

print(f"Lesion area       : {area:.0f} px²")
print(f"Lesion perimeter  : {perimeter:.1f} px")
print(f"Circularity       : {circularity:.3f}  (1.0 = perfect circle, <0.7 is clinically irregular)")


In [ ]:
# Draw the detected contour on the original image
contour_img = img_rgb.copy()
if contour is not None:
    cv2.drawContours(contour_img, [contour], -1, (0, 200, 50), 3)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(img_rgb);       axes[0].set_title("Original"); axes[0].axis("off")
axes[1].imshow(contour_img);   axes[1].set_title("Detected lesion contour (green)"); axes[1].axis("off")
plt.suptitle(f"Contour detection — circularity = {circularity:.3f}", fontsize=12)
plt.tight_layout()
plt.show()


---
##  SECTION 03 — HOW DOES A MODEL LEARN TO SEE? (CNNs)

The preprocessing pipeline is done. The Brong-Ahafo image arrives at the model as a clean, standardised 224×224 RGB tensor. Now the startup's engineer faces the central question of the entire project: how does the model learn to distinguish melanoma from seborrheic keratosis from healthy skin? What is actually happening inside this thing?

The answer is convolutional neural networks.


### The Convolution Operation

A convolution filter is a small template — typically 3×3 or 5×5 pixels — that slides across the image and computes a weighted sum at every position. The output is a **feature map**: a new image that shows how strongly the filter pattern matches at each location. A filter shaped like a vertical edge will produce high values wherever there is a vertical edge in the input, and near-zero values everywhere else.

A CNN stacks hundreds of these filters across tens or hundreds of layers. The key insight is that **the filters are learned from data**, not hand-designed. Given enough training examples, the network discovers which filter patterns are most useful for the task at hand.

Given an input image $X \in \mathbb{R}^{H \times W}$ and a filter $K \in \mathbb{R}^{k \times k}$, the output feature map at position $(i, j)$ is:

$$Y_{(i,j)} = \sum_{m=0}^{k-1} \sum_{n=0}^{k-1} X_{(i+m,\,j+n)} \cdot K_{(m,n)}$$

Each output value is a dot product between the filter and a $k \times k$ patch of the input — a measure of how similar the patch is to the filter pattern.

**Spatial dimensions.** Given input $H \times W$, filter size $k$, padding $p$, and stride $s$:

$$H_{\text{out}} = \left\lfloor \frac{H + 2p - k}{s} \right\rfloor + 1$$

Example: input 224×224, 3×3 filter, padding=1, stride=1 → $H_{\text{out}} = \lfloor (224 + 2 - 3) / 1 \rfloor + 1 = 224$. Padding preserves spatial dimensions.

**Depth.** Real CNN filters operate on all input channels simultaneously. A filter for a 3-channel RGB image has shape $3 \times k \times k$. The output is a single 2D feature map. $N$ filters produce $N$ feature maps, giving the next layer $N$ channels.

**What filters learn.** Early layers learn low-level features: edges, corners, colour blobs. Middle layers combine these into textures and patterns: scales, spots, fine networks of colour variation. Deep layers represent abstract concepts: "irregular border", "asymmetric colour distribution". This hierarchy of feature abstraction is why CNNs work for skin lesion detection.

**Pooling.** Max pooling takes the maximum value in a $k \times k$ region, reducing spatial dimensions while preserving the strongest feature activations. This introduces **translation invariance** — the lesion being slightly off-centre does not break the prediction, because the maximum of a region is the same regardless of which pixel in the region is the strongest.

**Receptive field.** The region of the original image that influences a single neuron at a given layer. A neuron in layer 1 sees only a 3×3 region of the input. A neuron in layer 5 sees a much larger region — the effect of combining multiple convolution layers. Receptive field growth is why deep networks can integrate global context.


In [ ]:
from scipy.ndimage import convolve

# Hand-crafted filters to demonstrate convolution before learning
FILTERS = {
    "Vertical edges (Sobel X)": np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32),
    "Horizontal edges (Sobel Y)": np.array([[-1,-2,-1], [0,0,0], [1,2,1]], dtype=np.float32),
    "Box blur (3×3)": np.ones((3,3), dtype=np.float32) / 9,
    "Laplacian sharpening": np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=np.float32),
}

img_gray_float = img_gray.astype(np.float32)

fig, axes = plt.subplots(2, 5, figsize=(22, 8))

# First column: the filter visualised
axes[0][0].imshow(img_gray, cmap="gray"); axes[0][0].set_title("Input image"); axes[0][0].axis("off")
axes[1][0].axis("off")

for col_idx, (filter_name, kernel) in enumerate(FILTERS.items(), start=1):
    result = convolve(img_gray_float, kernel)
    result_display = np.clip(result, 0, 255).astype(np.uint8)

    # Show the kernel as an annotated grid
    ax_k = axes[0][col_idx]
    im = ax_k.imshow(kernel, cmap="RdBu", vmin=-3, vmax=3)
    ax_k.set_title(filter_name, fontsize=9)
    for r in range(kernel.shape[0]):
        for c in range(kernel.shape[1]):
            ax_k.text(c, r, f"{kernel[r,c]:.1f}", ha="center", va="center", fontsize=8)
    ax_k.set_xticks([]); ax_k.set_yticks([])

    ax_r = axes[1][col_idx]
    ax_r.imshow(result_display, cmap="gray")
    ax_r.set_title("Output feature map", fontsize=9)
    ax_r.axis("off")

plt.suptitle("Convolution: each filter extracts a different feature from the same image", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# 🧪 EXPERIMENT — design your own filter
MY_FILTER = np.array([
    [ 0, -1,  0],
    [-1,  4, -1],
    [ 0, -1,  0]
], dtype=np.float32)
# This is a Laplacian filter — it highlights rapid intensity changes.
# Try these alternatives:
# Emboss:    np.array([[-2,-1,0],[-1,1,1],[0,1,2]])
# Diagonal:  np.array([[2,1,0],[1,0,-1],[0,-1,-2]])
# Identity:  np.array([[0,0,0],[0,1,0],[0,0,0]])

from scipy.ndimage import convolve
result = np.clip(convolve(img_gray.astype(np.float32), MY_FILTER), 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img_gray, cmap="gray"); axes[0].set_title("Input"); axes[0].axis("off")
axes[1].imshow(result,   cmap="gray"); axes[1].set_title("Your filter output"); axes[1].axis("off")
plt.tight_layout(); plt.show()
# What does your filter emphasise in the lesion image?


In [ ]:
# Load a pretrained EfficientNet-B0 and extract feature maps at different depths
# This shows the progression from edge features to abstract patterns

import timm
from utils.model_utils import load_efficientnet, get_image_tensor, extract_feature_maps

print("Loading EfficientNet-B0 (pretrained on ImageNet)...")
model = load_efficientnet(num_classes=1000, device=DEVICE)

pil_img    = Image.open(SAMPLE_IMAGE_PATH).convert("RGB")
img_tensor = get_image_tensor(pil_img, device=DEVICE)

# Extract feature maps from early, mid, and late blocks
layer_names = ["blocks.0", "blocks.2", "blocks.5"]
print("Extracting feature maps...")
feature_maps = extract_feature_maps(model, img_tensor, layer_names)

print("Feature map shapes:")
for name, fmap in feature_maps.items():
    print(f"  {name}: {fmap.shape}  (channels × H × W)")


In [ ]:
# Visualise feature maps — the progression from low-level to abstract features
fig, axes_grid = plt.subplots(len(layer_names), 9, figsize=(20, 7))

for row_idx, (layer_name, fmap) in enumerate(feature_maps.items()):
    # Show first 8 channels
    n_show = min(8, fmap.shape[0])
    axes_grid[row_idx][0].imshow(img_rgb)
    axes_grid[row_idx][0].set_title("Input", fontsize=8)
    axes_grid[row_idx][0].axis("off")

    for ch_idx in range(n_show):
        channel = fmap[ch_idx]
        channel_norm = (channel - channel.min()) / (channel.max() - channel.min() + 1e-8)
        axes_grid[row_idx][ch_idx + 1].imshow(channel_norm, cmap="viridis")
        axes_grid[row_idx][ch_idx + 1].set_title(f"Ch {ch_idx}", fontsize=7)
        axes_grid[row_idx][ch_idx + 1].axis("off")

    axes_grid[row_idx][0].set_ylabel(f"{layer_name}{fmap.shape}", fontsize=8, rotation=0,
    labelpad=60, va="center")

plt.suptitle("EfficientNet-B0 feature maps — early layers detect edges, deep layers detect patterns",
             fontsize=11)
plt.tight_layout()
plt.show()


### Backpropagation and How CNNs Learn

**The loss function.** At the end of the forward pass, the model outputs a probability distribution over classes. The **cross-entropy loss** measures how wrong this distribution is:

$$\mathcal{L} = -\log(\hat{p}_c)$$

where $\hat{p}_c$ is the predicted probability for the correct class $c$. If the model is confident and correct (e.g., $\hat{p}_c = 0.95$), the loss is small ($-\log(0.95) \approx 0.05$). If the model is confident and wrong ($\hat{p}_c = 0.01$), the loss is large ($-\log(0.01) \approx 4.6$). The model is punished proportionally to how wrong and how certain it was.

**Gradient descent.** The filter weights are updated by moving them slightly in the direction that reduces the loss:

$$w \leftarrow w - \eta \nabla_w \mathcal{L}$$

where $\eta$ is the **learning rate** — a small positive number controlling the step size. Too small: learning is painfully slow. Too large: the updates overshoot and the loss oscillates instead of decreasing.

**Backpropagation** is the chain rule applied recursively from the loss back through every layer. The gradient of the loss with respect to each filter weight tells that filter: "move in this direction to reduce the error on this training example." Over thousands of training steps, the filters self-organise. The model was not told what a lesion boundary looks like — it discovered that edge-like features are useful, because images with clear edges tend to be classified correctly when those features are present.

**Training vs. inference.** During training, gradients flow *backward* through all layers — this is computationally expensive and requires storing activations at every layer. During inference, only the *forward pass* runs — much faster. The `model.eval()` call disables dropout and batch normalisation updates; `torch.no_grad()` prevents PyTorch from building the computational graph for backpropagation. Neither is optional.

**Grad-CAM (Gradient-weighted Class Activation Mapping)** is a technique that uses the gradients of the classification score with respect to the last convolutional layer to produce a heatmap showing *which regions of the image most influenced the prediction*. It gives us a visual answer to the question "what is the model actually looking at?"


In [ ]:
from utils.model_utils import predict_class, compute_gradcam

# Run a forward pass and display the prediction
# Using ImageNet classes
import json, urllib.request

# Load ImageNet class names
try:
    url = "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json"
    with urllib.request.urlopen(url, timeout=5) as r:
        class_names = json.loads(r.read())
except Exception:
    class_names = [f"class_{i}" for i in range(1000)]

result = predict_class(model, img_tensor, class_names)

print(f"Top prediction : {result['predicted']}  ({result['confidence']:.1%} confidence)")
print()
print("Top-5 predictions:")
top5 = sorted(result["all"].items(), key=lambda x: x[1], reverse=True)[:5]
for name, prob in top5:
    bar = "█" * int(prob * 40)
    print(f"  {name:<30s} {prob:.1%}  {bar}")


In [ ]:
from utils.model_utils import compute_gradcam

# Compute Grad-CAM for the top predicted class
top_class_idx = class_names.index(result["predicted"]) if result["predicted"] in class_names else 0

print("Computing Grad-CAM heatmap...")
heatmap = compute_gradcam(
    model, img_tensor,
    target_class=top_class_idx,
    target_layer_name="blocks.5"
)

# Overlay heatmap on original image
heatmap_resized = cv2.resize(heatmap, (img_rgb.shape[1], img_rgb.shape[0]))
heatmap_colour  = cv2.applyColorMap((heatmap_resized * 255).astype(np.uint8), cv2.COLORMAP_JET)
heatmap_colour  = cv2.cvtColor(heatmap_colour, cv2.COLOR_BGR2RGB)
overlay         = (img_rgb.astype(np.float32) * 0.55 + heatmap_colour.astype(np.float32) * 0.45).clip(0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_rgb);            axes[0].set_title("Original image");     axes[0].axis("off")
axes[1].imshow(heatmap_resized, cmap="jet"); axes[1].set_title("Grad-CAM heatmap"); axes[1].axis("off")
axes[2].imshow(overlay);            axes[2].set_title(f"Overlay{result['predicted']} ({result['confidence']:.1%})"); axes[2].axis("off")
plt.suptitle("Grad-CAM — what the model is 'looking at' to make its prediction", fontsize=12)
plt.tight_layout()
plt.show()
print("Hot areas (red/yellow) had the strongest influence on the classification.")
print("If the model looks at the background instead of the lesion, that is a failure mode.")


### ResNet and EfficientNet: Why Architecture Matters

Convolutional networks can be made deeper — more layers, more abstract features. But deeper networks have a problem: during backpropagation, gradients must flow from the last layer all the way back to the first. At each layer, the gradient is multiplied by a weight matrix. If those weights are slightly less than 1 in magnitude, the gradient shrinks exponentially with depth — **the vanishing gradient problem**. Early layers receive almost no gradient signal and stop learning entirely.

**Residual connections (ResNet, He et al. 2015)** solve this with a single elegant addition: a skip connection that adds the input of a block to its output:

$$y = F(x) + x$$

The gradient of the loss with respect to $x$ now includes a direct path through the addition — it does not have to pass through $F(x)$ at all. This single innovation made networks with 100, 152, even 1000 layers trainable. Today, almost every serious CNN architecture includes skip connections.

**EfficientNet (Tan & Le, 2019)** asks a different question: given a fixed computational budget, what is the best way to allocate it? Should you make the network deeper (more layers), wider (more channels per layer), or use higher input resolution? The authors showed that **compound scaling** — increasing all three dimensions together under a fixed budget — consistently outperforms scaling any single dimension. EfficientNet-B0 achieves higher accuracy than ResNet-50 with 5.3× fewer parameters.


---
📚 **Further Reading & Watching**

| Resource | Type | Why it's worth your time |
|---|---|---|
| [But what is a neural network? — 3Blue1Brown](https://www.youtube.com/watch?v=aircAruvnKk) | 📹 Video (19 min) | The best visual intuition for neural networks that exists. Watch this first. |
| [Gradient descent, how neural networks learn — 3Blue1Brown](https://www.youtube.com/watch?v=IHZwWFHWa-w) | 📹 Video (21 min) | Explains backpropagation visually before the math. Essential. |
| [CNN Explainer — Polo Club of Data Science](https://poloclub.github.io/cnn-explainer/) | 🖥️ Interactive | Watch feature maps update as you change the input image |
| [Feature Visualization — Distill.pub](https://distill.pub/2017/feature-visualization/) | 📄 Article | Stunning visual essay on what CNN filters actually learn |
| [CS231n — Convolutional Neural Networks](https://cs231n.github.io/convolutional-networks/) | 📖 Notes | Stanford's definitive written reference on CNNs |
| [EfficientNet paper — Tan & Le (2019)](https://arxiv.org/abs/1905.11946) | 📄 Paper | The paper behind the model we use |

---


In [ ]:
# Print EfficientNet-B0 architecture summary
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("EfficientNet-B0 Architecture Summary")
print("=" * 50)
print(f"Total parameters   : {total_params:,}")
print(f"Trainable params   : {trainable:,}")
print(f"Approx. size       : {total_params * 4 / 1e6:.1f} MB (float32)")
print()
print("Top-level modules:")
for name, module in model.named_children():
    n_params = sum(p.numel() for p in module.parameters())
    print(f"  {name:<20s} {n_params:>10,} params")


## SECTION 04 — WHICH MODEL? YOLO, VISION TRANSFORMERS, AND MULTIMODAL MODELS

---

The startup's CTO has a new requirement. The app is not just classifying images — it is flagging specific lesions and drawing bounding boxes around them so the community health worker knows exactly which area to refer for specialist review. Classification (one label for the whole image) is not enough. We need detection. Enter YOLO.

And while we are here, we need to ask the bigger question: is a CNN even the right architecture for this task? Can attention-based models do what convolution cannot?


### YOLO: Single-Pass Object Detection

**The detection problem.** Unlike classification, detection must simultaneously predict:
1. *What* objects are present (class labels)
2. *Where* each object is (bounding box coordinates: $x, y, w, h$)
3. *How confident* the prediction is (objectness score)

And it must do this for multiple objects of varying sizes in a single image.

**The naive approach — sliding window** — runs a classifier at every position and scale: a 100×100 window at position (0,0), then (1,0), then (2,0)... at 50 scales... across a 224×224 image. This is computationally intractable for real-time use.

**YOLO's insight:** divide the image into an $S \times S$ grid. Each cell predicts $B$ bounding boxes with confidence scores and $C$ class probabilities — *all in a single forward pass*. For a 7×7 grid with 2 boxes and 80 classes, the output tensor has shape $7 \times 7 \times (2 \times 5 + 80) = 7 \times 7 \times 90$. The entire detection problem is reframed as a single regression task.

**Anchor-free detection (YOLOv8).** Modern YOLO versions remove anchor boxes entirely. The network directly predicts box coordinates as offsets from grid cell centres. This simplifies training and improves performance on objects of unusual aspect ratios — relevant for lesions, which can be elongated or irregularly shaped.

**Non-maximum suppression (NMS).** Multiple grid cells may each predict a box for the same lesion. NMS resolves this:
1. Sort all predicted boxes by confidence score (descending)
2. Keep the highest-confidence box
3. Remove all other boxes with **IoU (Intersection over Union)** > threshold with the kept box
4. Repeat for remaining boxes

$$\text{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

IoU of 1.0 means perfect overlap; IoU of 0 means no overlap. An NMS threshold of 0.5 keeps boxes that overlap the selected box by less than 50%.

**Speed vs. accuracy.** YOLOv8n (nano): ~200 FPS on GPU, fast enough for real-time mobile use. YOLOv8x (extra-large): ~50 FPS but significantly higher mAP. For a community health app on a Tecno Spark phone, the nano variant is the realistic choice — and it is still remarkably capable.

---
📚 **Further Reading & Watching**

| Resource | Type | Why it's worth your time |
|---|---|---|
| [You Only Look Once — Redmon et al. (2015)](https://arxiv.org/abs/1506.02640) | 📄 Paper | The original YOLO paper. The introduction explains single-pass detection clearly. |
| [Ultralytics YOLOv8 Documentation](https://docs.ultralytics.com/) | 📖 Docs | Official docs for the version we use. Excellent quick-start guides. |
| [YOLO Object Detection Explained — Aleksa Gordić](https://www.youtube.com/watch?v=ag3DLKsl2vk) | 📹 Video (30 min) | Technical but accessible walkthrough of YOLO architecture evolution |

---


In [ ]:
from utils.model_utils import load_yolo

print("Loading YOLOv8n (nano variant)...")
yolo = load_yolo(model_size="n", device=DEVICE)

# Run detection on the sample image
print("Running detection...")
results = yolo(SAMPLE_IMAGE_PATH, verbose=False)

# Display annotated result
if results and len(results) > 0:
    annotated = results[0].plot()
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(img_rgb);         axes[0].set_title("Original"); axes[0].axis("off")
    axes[1].imshow(annotated_rgb);   axes[1].set_title("YOLOv8n detections"); axes[1].axis("off")
    plt.suptitle("YOLOv8n object detection — single forward pass", fontsize=12)
    plt.tight_layout()
    plt.show()

    # Show raw detection data
    boxes = results[0].boxes
    print(f"\nDetections found: {len(boxes)}")
    for i, box in enumerate(boxes):
        print(f"  Box {i+1}: xyxy={box.xyxy[0].cpu().numpy().round(1)}, "
              f"conf={float(box.conf[0]):.2f}, cls={int(box.cls[0])}")
else:
    print("No detections on this image — the model is pretrained on COCO classes (people, cars, etc.)")
    print("For lesion detection, YOLOv8 would be fine-tuned on a skin lesion dataset.")


In [ ]:
# 🧪 EXPERIMENT — adjust detection thresholds
CONFIDENCE_THRESHOLD = 0.25   # try 0.1 (more detections), 0.5 (fewer, higher confidence)
IOU_THRESHOLD        = 0.45   # NMS IoU threshold — try 0.1, 0.7

results_exp = yolo(SAMPLE_IMAGE_PATH, conf=CONFIDENCE_THRESHOLD,
                   iou=IOU_THRESHOLD, verbose=False)

if results_exp:
    annotated_exp = cv2.cvtColor(results_exp[0].plot(), cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.imshow(annotated_exp)
    ax.set_title(f"conf>={CONFIDENCE_THRESHOLD}, IoU NMS={IOU_THRESHOLD}")
    ax.axis("off"); plt.tight_layout(); plt.show()

    print(f"Detections with conf>={CONFIDENCE_THRESHOLD}: {len(results_exp[0].boxes)}")
# Lower confidence → more detections, but more false alarms.
# Higher IoU threshold → fewer boxes merged, more overlapping detections kept.


### Vision Transformers: Attention Over Convolution

CNNs are powerful. But they have a fundamental limitation: a convolutional filter has a **fixed receptive field**. To capture a relationship between a lesion at position $(10, 200)$ and a lesion at $(180, 30)$, the network must compose many layers of convolution until the receptive field grows large enough to encompass both. Global context is built slowly, layer by layer.

**The transformer insight:** attention allows every position in the image to directly attend to every other position in a single operation, regardless of spatial distance. Global context is captured in the first layer.

**Patches as tokens (ViT).** Divide the image into fixed-size patches — 16×16 pixels is standard. Flatten each patch and linearly project it to a $d$-dimensional embedding. A 224×224 image with 16×16 patches yields 196 tokens. This sequence of patch embeddings is the input to the transformer — exactly analogous to word embeddings in NLP.

**Self-attention.** Given a sequence of token embeddings $X$, the model computes:

$$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$$

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Each token forms a **query** ($Q$ — what am I looking for?), and all tokens have **keys** ($K$ — what do I contain?) and **values** ($V$ — what do I communicate if attended to?). The dot product $QK^T$ measures compatibility between every query-key pair. The softmax turns these into a probability distribution over all tokens. The output is a weighted sum of values, with higher weight for more compatible keys.

The $\sqrt{d_k}$ scaling prevents dot products from growing too large as embedding dimension increases, which would push the softmax into a region of very small gradients.

**Multi-head attention** runs $h$ attention heads in parallel, each with different learned $W_Q, W_K, W_V$ matrices. Different heads can attend to different aspects simultaneously: one head might focus on colour variation, another on boundary shape, another on textural regularity.

**Position embeddings.** Unlike CNNs, transformers have no inherent notion of spatial position — attention is permutation-invariant. Learnable position embeddings are added to each patch embedding to encode where in the image each patch came from.

**When do transformers beat CNNs?** Generally when: (1) training data is large enough to learn position embeddings and multi-head attention from scratch; (2) the task requires long-range spatial reasoning; (3) global understanding matters more than local texture. For skin lesion detection at scale, Vision Transformers match or exceed CNN performance when pretrained on large datasets.

---
📚 **Further Reading & Watching**

| Resource | Type | Why it's worth your time |
|---|---|---|
| [The Illustrated Transformer — Jay Alammar](https://jalammar.github.io/illustrated-transformer/) | 📄 Article | The single best visual explanation of how transformers work |
| [An Image is Worth 16×16 Words — Dosovitskiy et al. (2020)](https://arxiv.org/abs/2010.11929) | 📄 Paper | The Vision Transformer paper — the patch-as-token intuition is elegant |
| [Attention Is All You Need — Vaswani et al. (2017)](https://arxiv.org/abs/1706.03762) | 📄 Paper | The paper that started it all |
| [Andrej Karpathy — Building attention from scratch](https://www.youtube.com/watch?v=PaCmpygFfXo) | 📹 Video (2 hr) | If you want to truly understand self-attention, build it |

---


In [ ]:
from utils.model_utils import load_vit

print("Loading ViT-B/16 (pretrained on ImageNet)...")
vit = load_vit(model_name="vit_base_patch16_224", device=DEVICE)
vit_params = sum(p.numel() for p in vit.parameters())

vit_result = predict_class(vit, img_tensor, class_names)
print(f"ViT-B/16 prediction  : {vit_result['predicted']}  ({vit_result['confidence']:.1%})")
print(f"EfficientNet-B0 pred : {result['predicted']}  ({result['confidence']:.1%})")
print()
print(f"ViT-B/16 parameters      : {vit_params:,}")
print(f"EfficientNet-B0 params   : {total_params:,}")
print(f"Size ratio               : {vit_params / total_params:.1f}×  (ViT is larger)")


In [ ]:
# Visualise patch tokenisation — how ViT 'sees' the image
PATCH_SIZE = 16
H, W = 224, 224
n_patches_h, n_patches_w = H // PATCH_SIZE, W // PATCH_SIZE

# Resize the image to 224×224 first
img_224 = cv2.resize(img_rgb, (224, 224))

fig, axes = plt.subplots(n_patches_h, n_patches_w + 1,
                          figsize=(n_patches_w + 1, n_patches_h))
axes[0][0].imshow(img_224); axes[0][0].set_title("Input", fontsize=7); axes[0][0].axis("off")
for r in range(1, n_patches_h): axes[r][0].axis("off")

for r in range(n_patches_h):
    for c in range(n_patches_w):
        patch = img_224[r*PATCH_SIZE:(r+1)*PATCH_SIZE, c*PATCH_SIZE:(c+1)*PATCH_SIZE]
        axes[r][c + 1].imshow(patch)
        axes[r][c + 1].axis("off")

plt.suptitle(f"ViT patch tokenisation: {n_patches_h}×{n_patches_w} = {n_patches_h*n_patches_w} tokens "
             f"(patch size {PATCH_SIZE}×{PATCH_SIZE})", fontsize=10)
plt.tight_layout(pad=0.1)
plt.show()


In [ ]:
# 🧪 EXPERIMENT — different patch sizes
PATCH_SIZE_EXP = 32   # try 8 (784 tokens, fine detail), 16 (196), 32 (49 tokens, coarser)

n_h = 224 // PATCH_SIZE_EXP
n_w = 224 // PATCH_SIZE_EXP

# Draw patch grid over the image
grid_img = img_224.copy()
for i in range(0, 224, PATCH_SIZE_EXP):
    cv2.line(grid_img, (i, 0), (i, 224), (0, 180, 90), 1)
    cv2.line(grid_img, (0, i), (224, i), (0, 180, 90), 1)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img_224);  axes[0].set_title("Original 224×224"); axes[0].axis("off")
axes[1].imshow(grid_img); axes[1].set_title(f"Patch grid: {n_h}×{n_w} = {n_h*n_w} tokens (size {PATCH_SIZE_EXP}px)")
axes[1].axis("off")
plt.tight_layout(); plt.show()
# With patch_size=8: 784 tokens — finer spatial resolution, much more compute
# With patch_size=32: 49 tokens — coarser, faster, may miss fine lesion detail


### Multimodal Models: CLIP, LLaVA, and Medical Vision-Language Models

**The multimodal insight.** Vision and language are not separate problems. Images and text describe the same world. A model that understands both can reason across the boundary between them — classifying an image by comparing it to text descriptions, or explaining a visual finding in natural language.

**CLIP (Contrastive Language-Image Pretraining, Radford et al. 2021)** was trained on 400 million image-text pairs scraped from the web. It learned a **shared embedding space** where images and their text descriptions are close together, and unrelated image-text pairs are far apart.

The training objective: for a batch of $N$ image-text pairs, maximise the similarity of the $N$ matching pairs while minimising the $N^2 - N$ non-matching pairs. This is computed as a symmetric cross-entropy over the cosine similarity matrix.

**Zero-shot inference** requires no fine-tuning. Encode the image. Encode text prompts like *"a dermoscopic image of melanoma"* and *"a benign skin mole"*. Rank the prompts by cosine similarity to the image embedding. The highest-similarity prompt is the zero-shot prediction.

This is powerful for skin lesion analysis: you can adapt the classification to a new condition simply by changing the text prompts — no labelled training data required.

**LLaVA (Large Language and Vision Assistant)** connects a CLIP vision encoder to a large language model via a linear projection layer. The model can answer questions about images in natural language: *"What features in this image suggest the lesion is irregular?"* A community health worker who doesn't know what "0.87 melanoma confidence" means can act on *"The border of this lesion is irregular and the pigmentation is uneven. Recommend specialist review."*

**The clinical opportunity and limitation.** Explainability in language is fundamentally more deployable in a clinical context than a probability score. But multimodal models are large — LLaVA-7B requires a GPU. For the Brong-Ahafo context, edge deployment is not yet practical at scale. What *is* practical today: CLIP zero-shot on a server, with the community health worker uploading the image over mobile data.

---
📚 **Further Reading & Watching**

| Resource | Type | Why it's worth your time |
|---|---|---|
| [Learning Transferable Visual Models From Natural Language — Radford et al. (2021)](https://arxiv.org/abs/2103.00020) | 📄 Paper | The CLIP paper. Section 2 (approach) is clear and worth reading in full. |
| [Visual Instruction Tuning (LLaVA) — Liu et al. (2023)](https://arxiv.org/abs/2304.08485) | 📄 Paper | How CLIP vision encoders are connected to language models |
| [Gemini: A Family of Highly Capable Multimodal Models](https://arxiv.org/abs/2312.11805) | 📄 Paper | Section 3 explains multimodal architecture and medical imaging benchmarks |

---


In [ ]:
from utils.model_utils import load_clip, clip_zero_shot

print("Loading CLIP (ViT-B/32)...")
clip_model, clip_processor = load_clip(device=DEVICE)

# Zero-shot classification with clinical prompts
prompts = [
    "a dermoscopic image of melanoma",
    "a benign melanocytic nevus (mole)",
    "a basal cell carcinoma",
    "a seborrheic keratosis",
    "a normal healthy skin region",
]

print("Running CLIP zero-shot classification...")
clip_result = clip_zero_shot(clip_model, clip_processor, pil_img, prompts, device=DEVICE)

print("\nCLIP zero-shot results:")
print("-" * 60)
for prompt, prob in sorted(clip_result.items(), key=lambda x: x[1], reverse=True):
    bar = "█" * int(prob * 50)
    print(f"{prob:.1%}  {bar}\n       {prompt}\n")


In [ ]:
# 🧪 EXPERIMENT — write your own clinical text prompts
PROMPTS_EXP = [
    "a dermoscopic image of melanoma with irregular borders",
    "a benign skin mole with regular borders",
    "a skin lesion with asymmetric pigmentation",
    "a skin lesion on dark skin",
    # Add your own — try clinical language vs. everyday language:
    "a dark spot on skin",
]

results_exp = clip_zero_shot(clip_model, clip_processor, pil_img, PROMPTS_EXP, device=DEVICE)

print("CLIP results with custom prompts:")
for prompt, prob in sorted(results_exp.items(), key=lambda x: x[1], reverse=True):
    print(f"  {prob:.1%}  {prompt}")
# Try: does phrasing affect confidence?
# Does "dark skin" in the prompt change the result?


In [ ]:
# Compare CLIP confidence across skin tones — the zero-shot bias test
clinical_prompts = [
    "a skin lesion that needs medical attention",
    "a benign skin condition",
    "a normal skin region",
]

results_by_image = {}
for img_name, img_path in [
    ("Light skin (Fitzpatrick II)", light_path),
    ("Dark skin (Fitzpatrick V)",  dark_path),
]:
    if Path(img_path).exists():
        pil_test = Image.open(img_path).convert("RGB")
        results_by_image[img_name] = clip_zero_shot(
            clip_model, clip_processor, pil_test, clinical_prompts, device=DEVICE
        )
    else:
        # Use main image as placeholder
        results_by_image[img_name] = clip_zero_shot(
            clip_model, clip_processor, pil_img, clinical_prompts, device=DEVICE
        )

print("CLIP confidence by skin tone (same prompts, same thresholds):")
print()
for img_name, res in results_by_image.items():
    print(f"  {img_name}:")
    for p, prob in sorted(res.items(), key=lambda x: x[1], reverse=True):
        print(f"    {prob:.1%}  {p}")
    print()
print("If confidence differs across skin tones for the same condition — that is a zero-shot bias.")



## SECTION 05 — WHERE VISION SYSTEMS ARE DEPLOYED
---

Understanding where computer vision is deployed and how it fails is essential context for auditing any specific system. The Accra startup is not building in isolation. It is building in a landscape where the same techniques are used to detect tumours, guide autonomous vehicles, try on clothes, inspect bridges, and monitor crowds. The stakes differ. The failure modes differ. The ethical pressures differ. Understanding the landscape matters for understanding where our app sits within it.


### Domain 1 — Medical Imaging

Medical imaging is where computer vision has the clearest potential to do good — and the clearest potential to cause harm at scale.

**The technical landscape.** Clinical vision tasks span three levels of analysis: *classification* (is this chest X-ray normal or abnormal?), *detection* (where are the suspect lymph nodes in this CT scan?), and *segmentation* (delineate the exact tumour boundary pixel by pixel for radiation planning). The same three tasks as any vision system — but the labels come from radiologists, pathologists, and dermatologists, and the margin for error is measured in lives.

**Domain shift** is the most dangerous failure mode. A model trained on dermatoscope images — standardised, high-resolution, carefully controlled — will fail on phone-camera images, not because the task changed, but because the input distribution changed. Performance drops can be catastrophic and invisible: the model still outputs a confidence score, but that confidence score is no longer calibrated to reality.

**Regulatory context.** In the US and EU, AI models used for clinical decision support are regulated as Software as a Medical Device (SaMD). The FDA requires pre-market notification for AI-powered diagnostic tools. Ghana does not yet have an equivalent regulatory framework for AI medical devices which means the startup is operating in a regulatory vacuum. This is not an excuse to deploy carelessly; it is an argument for applying voluntary standards more rigorously.

**The ABCDE criteria.** Dermatologists use the ABCDE mnemonic to screen skin lesions for melanoma:
- **A**symmetry — one half does not match the other
- **B**order irregularity — ragged, notched, or blurred edges
- **C**olour variation — multiple shades of brown, black, red
- **D**iameter — larger than 6mm (a pencil eraser)
- **E**volution — changing in size, shape, or colour over time

These criteria translate directly to computational features: asymmetry → contour comparison across axes; border → circularity and fractal dimension; colour → entropy of the HSV histogram. The model is, in a sense, learning a non-linear function of these ABCDE features from data.



### Domain 2 — Autonomous Vehicles

Self-driving vehicles are the highest-stakes deployment of computer vision outside of medicine. The system must detect pedestrians, vehicles, lane markings, traffic lights, and road hazards in real time — in rain, at night, in direct sunlight, at highway speeds.

**Sensor fusion** is the dominant architecture: cameras provide rich colour and texture information; LiDAR provides precise 3D depth; radar provides velocity and works in fog and rain where cameras fail. The fusion happens at the feature level — a unified spatial representation that combines the best of each sensor. Camera-only systems (like early Tesla Autopilot) are cheaper but more vulnerable to adverse conditions.

**Real-time requirements** are non-negotiable. The full perception stack — detection, tracking, depth estimation, lane segmentation — must complete in under 100ms on a moving vehicle. This eliminates large transformer models (ViT-L on CPU takes ~500ms) and favours architectures that trade accuracy for speed: EfficientDet, YOLO, and dedicated neural processing units (NPUs) in automotive chips.

**The long-tail problem.** Training data covers common scenarios well: daylight, good weather, urban/suburban roads. But safety depends on *rare* events: a child running into the road from between parked cars at night. These events are underrepresented by definition — rare in training data because they are rare in the world. But they are the exact scenarios where the cost of failure is highest.

> ⚖️ **Ethical Lens:** African road conditions are systematically underrepresented in autonomous vehicle training data: unpaved roads, informal traffic patterns, motorbike-heavy traffic, livestock, night markets. A system validated on US and European roads will encounter distribution shift on Accra's roads, and the failure modes may be different from what the developers anticipated. Who was in the room when the long-tail edge cases were defined?


### Domain 3 — Fashion and Virtual Try-On

Fashion is the lightest-stakes application in this list — and the one that reveals how quickly bias accumulates even in low-stakes settings.

**The task.** Virtual try-on systems allow customers to see how a garment would look on their body without physically trying it on. The pipeline requires: (1) human pose estimation — detect 17+ body keypoints (shoulders, hips, knees, etc.); (2) garment segmentation — separate the clothing from the background; (3) warping — deform the garment image to fit the predicted body shape; (4) blending — compositing the warped garment onto the person image.

**VITON (Virtual Try-On Network)** introduced a two-stage architecture that remains the dominant approach: a coarse warping stage using a geometric transformation network, followed by a refinement stage using a U-Net generator. More recent systems use diffusion models for photorealistic output.

**Biometric data and consent.** In-store virtual try-on kiosks capture 3D body scans. These are biometric data under GDPR and similar frameworks — they uniquely identify individuals and cannot be anonymised. Retention policies are rarely disclosed to customers at the point of capture.

> ⚖️ **Ethical Lens:** Virtual try-on systems trained on datasets with limited body-type diversity produce systematically worse results on bodies outside the training distribution. Dark skin tones reflect light differently from lighter skin.


### Domain 4 — Civil Engineering and Infrastructure

Structural inspection is an application of computer vision that is largely invisible to the public but affects everyone who uses a bridge, a dam, or a building.

**Crack detection** from drone or satellite imagery treats the problem as pixel-level binary classification: each pixel is either crack or not crack. The challenge is that cracks are visually similar to joints, expansion gaps, and surface staining — and the model must distinguish them from 50m in the air. Convolutional segmentation architectures (U-Net, DeepLab) are the standard approach.

**Optical flow** measures the apparent motion of visual features between frames — effectively estimating which parts of a structure are moving relative to the camera. Applied to a bridge under load, optical flow can detect deflection patterns that precede structural failure.

**Flood and disaster mapping** from multispectral satellite imagery uses the near-infrared channel (invisible to human eyes, but distinguishable by satellites) to identify flooded areas based on spectral signatures. The Sentinel-2 and Landsat satellites provide free multispectral imagery at 10m resolution — high enough to map individual buildings.

> ⚖️ **Ethical Lens:** Infrastructure AI prioritises regions that are already well-mapped. Rural and peri-urban Africa is disproportionately absent from the training data for structural inspection models — because those regions have fewer bridges that have been systematically photographed and labelled. The result: AI-assisted inspection is available where inspection is already good, and absent where it is needed most.


### Domain 5 — Traffic and Surveillance

Traffic management and public safety surveillance represent the most ethically contested application of computer vision.

**The technical stack.** Vehicle detection and classification, automatic number plate recognition (ANPR), pedestrian counting, crowd density estimation. These are technically mature: YOLOv8 achieves near-human detection accuracy on vehicle datasets; ANPR achieves >99% accuracy in controlled conditions. The technical problems are largely solved.

**Re-identification across cameras** — tracking an individual across multiple camera feeds without using their face — is the frontier. Using clothing colour, body shape, and gait, re-id systems can follow a person through a city with no face recognition and no obvious identifier. This is technically impressive and democratically alarming.

**The documented disparities.** Buolamwini and Gebru (2018) — "Gender Shades" — showed that commercial face recognition systems misclassified darker-skinned women at rates up to 34 percentage points higher than lighter-skinned men. The same study catalysed a reckoning across the industry and led to the withdrawal of facial recognition products by IBM, Amazon, and Microsoft from law enforcement use.

> ⚖️ **Ethical Lens:** Surveillance infrastructure is being deployed in African cities — Nairobi, Accra, Lagos — often through export contracts with Chinese technology companies, at a pace that significantly outstrips the development of legislative frameworks for data governance. The consent question is not being asked. The right to challenge incorrect identification is not being protected. And the communities subject to the highest levels of surveillance are, in most cases, the same communities most vulnerable to the documented accuracy disparities. This is not a hypothetical risk.

---
📚 **Further Reading & Watching**

| Resource | Type | Why it's worth your time |
|---|---|---|
| [Detectron2 — Meta AI Blog](https://ai.meta.com/blog/detectron2-a-pytorch-based-modular-object-detection-library/) | 📄 Article | Good overview of detection, segmentation, and pose tasks |
| [Segment Anything Model — Meta AI](https://segment-anything.com/) | 🖥️ Demo | Interactive demo of instance segmentation |
| [Computer Vision Tasks Explained — Roboflow](https://blog.roboflow.com/computer-vision-tasks/) | 📄 Article | Clear taxonomy with visual examples of each task type |

---



##  SECTION 06 — ETHICS, DATASETS, AND WHAT COMES NEXT

---

We have built the technical picture. We know what an image is, how a preprocessing pipeline works, what tasks vision models perform, how CNNs and transformers learn, and where these systems are deployed. Now we ask the question the startup's CTO has been avoiding: is this specific model — trained on images from Austria and Australia — safe to deploy with community health workers in Ghana?




### Bias: Where It Enters and How to Measure It

Bias in machine learning is not a single thing. It enters the system at multiple stages, compounds through the pipeline, and is often invisible in aggregate metrics. A model reporting 90% accuracy can be simultaneously performing at 68% on the group whose health outcomes are most at stake.

**Data collection bias.** HAM10000 — the primary dermatology benchmark — was collected in Austria and Australia, using dermatoscopes, primarily from patients with Fitzpatrick Type I–III skin. The model trained on this data learns the distribution of this data. When deployed on Fitzpatrick Type V–VI images taken with phone cameras, it encounters a distribution it has never seen.

**Annotation bias.** Labels are human judgments. Annotator background influences labelling consistency. Conditions that present differently across skin tones (like erythema — redness — which is less visible on darker skin) may be systematically under-labelled in darker-skin images, not out of malice but out of unfamiliarity.

**Preprocessing bias.** As demonstrated in Section 02: Canny thresholds tuned on lighter-skin validation sets produce weaker feature representations for darker skin. This enters the model *before training begins*. The model is trained on systematically lower-quality representations of darker-skin images.

**Evaluation bias.** Reporting overall accuracy conceals per-group gaps. Consider a test set that is 85% Fitzpatrick Type I–III images: if the model achieves 92% accuracy on light skin and 68% on dark skin, the reported aggregate accuracy is $0.85 \times 0.92 + 0.15 \times 0.68 = 88.4\%$ — which sounds impressive. The 24-percentage-point gap is invisible.

**Fairness metrics.** Three formal definitions, each capturing a different notion of fairness:

- **Demographic parity:** $P(\hat{Y}=1 \mid A=a) = P(\hat{Y}=1 \mid A=b)$ — equal positive prediction rates across groups. A model that flags lesions at the same rate regardless of skin tone.

- **Equalized odds:** $P(\hat{Y}=1 \mid A=a, Y=y) = P(\hat{Y}=1 \mid A=b, Y=y)$ for both $y=0$ and $y=1$ — equal true positive rates *and* equal false positive rates across groups.

- **Calibration:** $P(Y=1 \mid \hat{p}=p, A=a) = p$ — confidence scores mean the same thing across groups. A model that says "80% melanoma" is right 80% of the time, whether the patient has light or dark skin.

**The impossibility theorem.** These three definitions cannot all be simultaneously satisfied when base rates differ between groups. This is not a solvable engineering problem — it is a values question about which type of fairness matters most in *this specific clinical context*. For a screening tool, equalized odds (equal sensitivity across groups) is almost certainly the right choice. Missing a melanoma on dark skin at a higher rate than on light skin is the failure mode that causes the most harm.

> ⚖️ **Ethical Lens:** The choice of fairness metric is a clinical and ethical decision, not a technical one. Engineers who make this choice implicitly — by optimising for overall accuracy — are making a values decision without acknowledging it.


### Key Datasets for African CV Work

A reference for practitioners working in this space.

**Dermatology:**
| Dataset | Size | Diversity | Access |
|---|---|---|---|
| [HAM10000](https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/DBW86T) | 10,015 images | Fitzpatrick I–III predominantly | Harvard Dataverse (free) |
| [Fitzpatrick17k](https://github.com/mattgroh/fitzpatrick17k) | 16,577 images | Explicit Fitzpatrick labels I–VI | GitHub (free) |
| [ISIC Archive](https://www.isic-archive.com/) | 70,000+ images | Limited dark-skin representation | Free account required |
| [SCIN (Google, 2024)](https://research.google/blog/scin-a-new-resource-for-representative-dermatology-images/) | 10,408 images | Diverse global skin tones | Free research access |

**General vision:**
| Dataset | Size | Use |
|---|---|---|
| [Open Images](https://storage.googleapis.com/openimages/web/index.html) | 9M images | Detection, segmentation, 600 classes |
| [COCO](https://cocodataset.org/) | 330K images | Detection, segmentation, keypoints |
| [Roboflow Universe](https://universe.roboflow.com/) | 100K+ datasets | Domain-specific annotated datasets |
| [HuggingFace Datasets](https://huggingface.co/datasets) | Growing | Standardised loading, all domains |

**Finding African-specific datasets:** search Google Dataset Search for "Africa" + your domain; check Papers With Code for African NLP and vision benchmarks; Makerere AI Health Lab (Uganda) has published medical imaging datasets from East African clinical settings.


### Model Cards and Datasheets: Documentation as Ethics

In 2018, two papers appeared that changed how responsible AI practitioners think about documentation.

**Datasheets for Datasets (Gebru et al., 2018)** proposed that every dataset should ship with a "datasheet" — structured documentation answering: How was the data collected? Who funded it? What are its known limitations? What populations are underrepresented? What should this dataset NOT be used for? The analogy is to electronic components: every component has a datasheet. Why not every dataset?

**Model Cards for Model Reporting (Mitchell et al., 2018)** proposed equivalent documentation for models: intended use cases, evaluation results broken down by subgroup, ethical considerations, limitations. A model card is the gap between "our model achieves 92% accuracy" (impressive-sounding) and "our model achieves 92% accuracy on Fitzpatrick I–III but has not been evaluated on Fitzpatrick V–VI" (honest).

Reading model cards before using a model, and writing them before deploying one, is the minimum ethical standard. The practice is spreading: HuggingFace prompts model card creation for every uploaded model; the FDA has incorporated model card concepts into its AI/ML guidance.


In [ ]:
# Generate a minimal model card for the EfficientNet-B0 in this tutorial
model_card = '''---
# Model Card: EfficientNet-B0 (ImageNet Pretrained)
# Used in: Building Ethical Vision Systems — Part 1 Tutorial

## Model Description
- **Architecture:** EfficientNet-B0 (Tan & Le, 2019)
- **Pretrained on:** ImageNet-1K (1.28M images, 1000 classes)
- **Parameters:** 5.3M
- **Input:** 224×224 RGB image, ImageNet normalisation
- **Output:** 1000-class probability distribution

## Intended Use
- Educational demonstration of CNN feature extraction and Grad-CAM
- NOT intended for clinical diagnosis or skin condition screening
- NOT validated on dermatology datasets

## Evaluation
- ImageNet top-1 accuracy: 77.1%
- **NOT evaluated on:** HAM10000, Fitzpatrick17k, ISIC, or any skin lesion dataset
- **NOT evaluated on:** Fitzpatrick Type V–VI skin tone images specifically

## Training Data
- ImageNet-1K: web-scraped images, majority from North America and Europe
- Skin representation: incidental — ImageNet contains some skin images but not as a primary category
- Demographic representation: unknown — ImageNet was not collected with demographic balance in mind

## Limitations and Risks
- Predictions on skin lesion images are NOT clinically validated
- Performance on dark skin tones (Fitzpatrick V–VI) is UNKNOWN
- Domain shift from ImageNet to clinical dermatology images is severe
- Should not be used as a decision-support tool in clinical settings without
  fine-tuning, clinical validation, and regulatory review

## Ethical Considerations
- This model should not be deployed as a screening tool for any patient population
  until validated on a demographically representative dataset
- Demographic parity and equalized odds have not been assessed

## How to Cite
    Tan, M., & Le, Q. V. (2019). EfficientNet: Rethinking Model Scaling for
    Convolutional Neural Networks. ICML 2019.
---'''

# Write the model card
card_path = Path("../assets/efficientnet_b0_model_card.md")
card_path.parent.mkdir(parents=True, exist_ok=True)
card_path.write_text(model_card)

print("Model card written to:", card_path.resolve())
print()
print(model_card[:800] + "...")



📚 **Complete Further Reading**

| Resource | Type | Why it's worth your time |
|---|---|---|
| [Datasheets for Datasets — Gebru et al. (2018)](https://arxiv.org/abs/1803.09010) | 📄 Paper | The paper that introduced dataset documentation as an ethical practice |
| [Model Cards for Model Reporting — Mitchell et al. (2018)](https://arxiv.org/abs/1810.03993) | 📄 Paper | The case for model documentation |
| [Racial Bias in Pulse Oximetry — Sjoding et al. (2020)](https://www.nejm.org/doi/full/10.1056/NEJMc2029240) | 📄 NEJM | The most cited real-world example of medical device bias on dark skin |
| [Coded Bias — Documentary (2020)](https://www.codedbias.com/) | 🎬 Film | Joy Buolamwini's investigation into facial recognition bias |
| [Timnit Gebru on AI Ethics and Representation](https://www.youtube.com/watch?v=I-EIVlHvHRM) | 📹 Video (45 min) | Directly relevant to AI in African contexts |
| [Dermatologist-level classification of skin cancer — Esteva et al., Nature](https://www.nature.com/articles/s41591-020-0842-3) | 📄 Paper | The landmark skin AI paper — and the one that largely ignored Fitzpatrick diversity |
| [AI in Africa — Barriers to AI Development (2022)](https://arxiv.org/abs/2201.01266) | 📄 Paper | Documents structural reasons African contexts are underrepresented in AI data |
| [Fitzpatrick17k Dataset](https://github.com/mattgroh/fitzpatrick17k) | 💾 Dataset | 17,000 skin images with explicit Fitzpatrick labels — the audit dataset |
| [SCIN Dataset — Google Research (2024)](https://research.google/blog/scin-a-new-resource-for-representative-dermatology-images/) | 💾 Dataset | Most diverse dermatology dataset available |
| [pytorch-image-models (timm)](https://github.com/huggingface/pytorch-image-models) | 🔧 Library | 700+ pretrained vision models in one library |
| [pytorch-grad-cam](https://github.com/jacobgil/pytorch-grad-cam) | 🔧 Library | Visualise what your CNN is looking at |

---

*Part 2 of this tutorial — led by Emmanuel Asante — performs a quantitative audit of a skin condition classifier on the HAM10000 dataset, measuring performance gaps across Fitzpatrick skin types and proposing mitigation strategies.*
